# AmsiNet - Step 23: a published-style script-identification baseline (A100, about 15-25 minutes)
**Why.** The paper compares against a script-identification method from the literature, not only generic classifiers. This step trains a **CNN + attention-based BiLSTM** in the style of Bhunia et al. (2019) on exactly the same word crops, split, ten classes and five seeds as Steps 2 and 16:
* an ImageNet-pretrained ResNet-18 encodes the 224x224 word crop; the feature map is pooled over height, so that each of its 7 columns is one step of a sequence along the width of the word;
* a bidirectional LSTM reads the sequence and an additive attention layer pools it into one vector, which a linear layer maps to the ten scripts.

This is a **reimplementation following the published description, trained under our protocol** (same loss, optimiser, 8 epochs, label smoothing 0.1; best epoch on validation macro-F1). The paper must describe it that way, not as a reproduction of the original code or numbers.

Probabilities are saved to `agents/flat_bhunia_seed{k}_*` in the same format as the other agents, and its cost to `agents/flat_bhunia_gflops.txt`. Afterwards run **Step 17 (re-run it; it now picks this model up)** on a CPU runtime and paste its summary table.

In [ ]:
# 1. Setup
from google.colab import drive
import os, subprocess, sys
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
try: import timm
except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=True); import timm
import json, glob, random, time, math, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from scipy import stats
from scipy.special import softmax
from sklearn.metrics import f1_score
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'; os.makedirs(LOCAL, exist_ok=True)
CFG = dict(SEEDS=[0, 1, 2, 3, 4], EPOCHS=8, BATCH=64, WORKERS=8)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SCRIPTS, INDEX = D['scripts'], D['index']; IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); META['y'] = META.script.map(SIDX).astype(int)
print(C, 'scripts:', SCRIPTS); print(META.groupby('split').size().to_dict())
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
assert os.path.exists(META.path.iloc[0]), 'crops not found on disk: ' + META.path.iloc[0]
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Model and training utilities

In [ ]:
import torch.nn.functional as F
class AttnBiLSTM(nn.Module):
    # CNN encoder -> sequence along the word width -> BiLSTM -> additive attention -> linear
    def __init__(self, n_classes, hidden=256):
        super().__init__()
        self.cnn = timm.create_model('resnet18.a1_in1k', pretrained=True, num_classes=0, global_pool='')   # feature map, 512 x 7 x 7
        self.rnn = nn.LSTM(512, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Sequential(nn.Linear(2 * hidden, 128), nn.Tanh(), nn.Linear(128, 1))
        self.drop = nn.Dropout(0.2); self.fc = nn.Linear(2 * hidden, n_classes)
    def forward(self, x):
        f = self.cnn(x).mean(2).transpose(1, 2)            # B x W x 512 (mean over height)
        h, _ = self.rnn(f.float())                          # B x W x 2H
        a = torch.softmax(self.att(h), dim=1)               # B x W x 1
        return self.fc(self.drop((a * h).sum(1)))
def make_model(name):
    return AttnBiLSTM(C) if name == 'bhunia' else timm.create_model(name, pretrained=True, num_classes=C)

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda'; FILL = (124, 116, 104)
class Crops(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]
def tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model.cnn if hasattr(model, 'cnn') else model); norm = T.Normalize(cfg['mean'], cfg['std'])
    return (T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL), T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm]), T.Compose([T.ToTensor(), norm]))
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def predict(model, df, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(df.path, df.y, tf), batch_size=256, num_workers=CFG['WORKERS']):
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
def mf1(P, y): return float(f1_score(y, P.argmax(1), average='macro', labels=list(range(C)), zero_division=0))
def train(tag, model_name, tr, va, epochs, lr, seed):
    # trains on dataframe tr, selects the epoch by macro-F1 on dataframe va, saves probabilities for ALL val and test words
    files = [f'{ROOT}/agents/{tag}_seed{seed}_{sp}_probs.npy' for sp in ('val', 'test')]
    if all(os.path.exists(f) for f in files): print('skip (done):', tag, seed); return
    set_seed(seed); model = make_model(model_name).to(DEV); ttf, etf = tfs(model)
    L = DataLoader(Crops(tr.path, tr.y, ttf), batch_size=CFG['BATCH'], shuffle=True, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = epochs * len(L); warm = max(1, int(0.05 * total)); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, state = -1, None
    for ep in range(epochs):
        model.train(); t0 = time.time(); rl = 0
        for x, y in L:
            x, y = x.to(DEV, non_blocking=True), torch.as_tensor(y).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
        Pv = predict(model, va, etf); f = mf1(Pv, va.y.values)
        print(f'{tag} seed{seed} ep{ep + 1}: loss {rl / len(L):.4f} val acc {(Pv.argmax(1) == va.y.values).mean():.4f} macro-F1 {f:.4f} ({time.time() - t0:.0f}s)')
        if f > best: best = f; state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state)
    for sp in ('val', 'test'):
        d = META[META.split == sp]; P = predict(model, d, etf); np.save(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_probs.npy', P.astype(np.float16)); pd.DataFrame({'path': d.path.values}).to_csv(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_paths.csv', index=False)
    print(f'done {tag} seed{seed}: best val macro-F1 {best:.4f}'); del model; torch.cuda.empty_cache()

## 3. Cost per word (FLOPs, two per multiply-accumulate)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
m = AttnBiLSTM(C).eval(); x = torch.zeros(1, 3, 224, 224)
with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
GF = fc.get_total_flops() / 1e9; params = sum(p.numel() for p in m.parameters()) / 1e6
print(f'flat_bhunia: {GF:.2f} GFLOPs per word, {params:.2f} M parameters')
open(f'{ROOT}/agents/flat_bhunia_gflops.txt', 'w').write(str(GF)); del m

## 4. Train (five seeds)

In [ ]:
TRN, VAL = META[META.split == 'train'], META[META.split == 'val']
for seed in CFG['SEEDS']: train('flat_bhunia', 'bhunia', TRN, VAL, CFG['EPOCHS'], 3e-4, seed)
print('training done')

## 5. Quick test-set check (not for model selection)

In [ ]:
TE = META[META.split == 'test']; yt = TE.y.values; a, f = [], []
for seed in CFG['SEEDS']:
    P = np.load(f'{ROOT}/agents/flat_bhunia_seed{seed}_test_probs.npy').astype(np.float32); a.append(100 * (P.argmax(1) == yt).mean()); f.append(100 * mf1(P, yt))
print(f'flat_bhunia: accuracy {np.mean(a):.2f} +/- {stats.t.ppf(0.975, len(a)-1) * np.std(a, ddof=1) / np.sqrt(len(a)):.2f} | macro-F1 {np.mean(f):.2f} +/- {stats.t.ppf(0.975, len(f)-1) * np.std(f, ddof=1) / np.sqrt(len(f)):.2f} (mean of {len(a)} seeds) | {GF:.2f} GFLOPs per word')
print('Now disconnect this runtime and re-run Step 17 on a CPU runtime.')